# 02 · Feature Engineering + Modelo ML — Motor 1

**Forecast de ventas por tienda × tipo de producto · 8 semanas**

### Qué hace este notebook
1. Lee los datos de Supabase y agrega a nivel semanal
2. Construye todas las features del modelo (tiempo, eventos, demanda histórica)
3. Entrena LightGBM con validación temporal
4. Mide el error (MAPE y MAE)
5. Genera el forecast de las próximas 8 semanas y lo guarda en Supabase

### Prerequisito
Haber corrido exitosamente `01_carga_datos.ipynb`

In [ ]:
import subprocess, sys
pkgs = ['pandas','numpy','lightgbm','scikit-learn',
        'sqlalchemy','psycopg2-binary','plotly','python-dotenv']
subprocess.run([sys.executable,'-m','pip','install','--quiet']+pkgs)
print('✅ Dependencias listas')

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pickle, os, sys, warnings
from pathlib import Path
from datetime import date
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sqlalchemy import text
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))
from conexion import get_engine

engine = get_engine()

## 1 · Cargar y agregar datos a nivel semanal

In [ ]:
print('Cargando datos de Supabase...')
df = pd.read_sql("""
    SELECT
        DATE_TRUNC('week', fecha)::date   AS semana,
        tienda_id, tipo_producto, familia,
        SUM(unidades_vendidas)                                AS unidades,
        SUM(valor_venta)                                      AS valor_venta,
        SUM(margen_bruto)                                     AS margen_bruto,
        AVG(descuento_pct)                                    AS descuento_pct_avg,
        AVG(CASE WHEN es_precio_pleno THEN 1.0 ELSE 0.0 END) AS pct_precio_pleno
    FROM fact_ventas_diarias
    GROUP BY DATE_TRUNC('week', fecha)::date, tienda_id, tipo_producto, familia
    ORDER BY semana, tienda_id, tipo_producto
""", engine)

df['semana'] = pd.to_datetime(df['semana'])
df_tiendas   = pd.read_sql('SELECT * FROM dim_tiendas', engine)
df_eventos   = pd.read_sql('SELECT * FROM dim_eventos', engine)
df_eventos['fecha'] = pd.to_datetime(df_eventos['fecha'])

print(f'✅ {len(df):,} filas | {df.tienda_id.nunique()} tiendas | {df.tipo_producto.nunique()} tipos')
print(f'   Rango: {df.semana.min().date()} → {df.semana.max().date()}')

## 2 · Feature Engineering

In [ ]:
# Join con perfil de tienda
df = df.merge(
    df_tiendas[['tienda_id','ciudad','formato','segmento_cliente','indice_rotacion','metros_cuadrados']],
    on='tienda_id', how='left'
)

# Features de tiempo
df['semana_iso']  = df['semana'].dt.isocalendar().week.astype(int)
df['mes']         = df['semana'].dt.month
df['anio']        = df['semana'].dt.year
df['trimestre']   = df['semana'].dt.quarter
df['es_quincena'] = df['semana'].dt.day.isin([14,15,16,28,29,30]).astype(int)
# Ciclicidad (captura estacionalidad sin saltos bruscos)
df['semana_sin']  = np.sin(2*np.pi*df['semana_iso']/52)
df['semana_cos']  = np.cos(2*np.pi*df['semana_iso']/52)
df['mes_sin']     = np.sin(2*np.pi*df['mes']/12)
df['mes_cos']     = np.cos(2*np.pi*df['mes']/12)
print('✅ Features de tiempo')

In [ ]:
# Lags y rolling — el modelo aprende de la demanda pasada
df = df.sort_values(['tienda_id','tipo_producto','semana'])
grp = df.groupby(['tienda_id','tipo_producto'])['unidades']

df['lag_1sem']      = grp.shift(1)
df['lag_2sem']      = grp.shift(2)
df['lag_4sem']      = grp.shift(4)
df['lag_8sem']      = grp.shift(8)
df['lag_52sem']     = grp.shift(52)   # mismo período año anterior
df['rolling_4sem']  = grp.shift(1).rolling(4, min_periods=1).mean().reset_index(0,drop=True)
df['rolling_8sem']  = grp.shift(1).rolling(8, min_periods=1).mean().reset_index(0,drop=True)
df['rolling_12sem'] = grp.shift(1).rolling(12,min_periods=1).mean().reset_index(0,drop=True)
df['rolling_std4']  = grp.shift(1).rolling(4, min_periods=1).std().reset_index(0,drop=True)
df['tendencia']     = (df['rolling_4sem'] / df['rolling_8sem'].replace(0,np.nan)).fillna(1)
print('✅ Features de demanda histórica (lags y rolling)')

In [ ]:
# Features de eventos — anticipación, evento y rebote
def crear_features_eventos(df, df_eventos):
    df = df.copy()
    for _, ev in df_eventos.iterrows():
        nombre   = ev['nombre_evento']
        fecha_ev = pd.to_datetime(ev['fecha'])
        alcance  = ev['alcance']
        pre      = int(ev['semanas_anticipacion'])
        post     = int(ev['semanas_rebote'])
        for s in range(-pre, post + 1):
            sem_obj = fecha_ev + pd.Timedelta(weeks=s)
            tag = f"ev_{nombre[:12]}" + (f"_pre{abs(s)}" if s<0 else "" if s==0 else f"_post{s}")
            if tag not in df.columns:
                df[tag] = 0
            mask = df['semana'] == sem_obj
            if alcance == 'nacional':
                df.loc[mask, tag] = 1
            else:
                df.loc[mask & (df['ciudad']==alcance), tag] = 1
    return df

df = crear_features_eventos(df, df_eventos)
cols_eventos = [c for c in df.columns if c.startswith('ev_')]
print(f'✅ {len(cols_eventos)} features de eventos (Días sin IVA, Navidad, Carnaval...)')

In [ ]:
# Encoding de variables categóricas
encoders = {}
for col in ['tipo_producto','familia','ciudad','formato','segmento_cliente']:
    enc = LabelEncoder()
    df[f'{col}_enc'] = enc.fit_transform(df[col].astype(str))
    encoders[col] = enc
print('✅ Encoding de categóricas listo')

## 3 · Entrenamiento del modelo LightGBM

In [ ]:
FEATURES = [
    'semana_iso','mes','trimestre','anio',
    'semana_sin','semana_cos','mes_sin','mes_cos','es_quincena',
    'lag_1sem','lag_2sem','lag_4sem','lag_8sem','lag_52sem',
    'rolling_4sem','rolling_8sem','rolling_12sem','rolling_std4','tendencia',
    'descuento_pct_avg','pct_precio_pleno',
    'tipo_producto_enc','familia_enc','ciudad_enc',
    'formato_enc','segmento_cliente_enc',
    'indice_rotacion','metros_cuadrados',
] + cols_eventos

# Split temporal — últimas 12 semanas como test
fecha_corte = df['semana'].max() - pd.Timedelta(weeks=12)
df_train = df[df['semana'] <= fecha_corte].dropna(subset=FEATURES)
df_test  = df[df['semana'] >  fecha_corte].dropna(subset=FEATURES)

X_train, y_train = df_train[FEATURES], df_train['unidades']
X_test,  y_test  = df_test[FEATURES],  df_test['unidades']

print(f'Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {len(FEATURES)}')

In [ ]:
modelo = lgb.LGBMRegressor(
    objective='regression', metric='mae',
    n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, verbose=-1
)
modelo.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)
print('✅ Modelo entrenado')

In [ ]:
y_pred = np.maximum(0, modelo.predict(X_test))
mape = mean_absolute_percentage_error(y_test[y_test>0], y_pred[y_test>0]) * 100
mae  = mean_absolute_error(y_test, y_pred)

print('='*45)
print('MÉTRICAS DE VALIDACIÓN')
print('='*45)
print(f'  MAPE : {mape:.1f}%  (referencia: <25% en moda)')
print(f'  MAE  : {mae:.1f} unidades / tienda × tipo × semana')
print('='*45)

# Importancia de features
imp = pd.DataFrame({'feature':FEATURES,'importance':modelo.feature_importances_})\
        .sort_values('importance',ascending=False).head(15)
fig = px.bar(imp, x='importance', y='feature', orientation='h',
             title='Top 15 features más importantes',
             color='importance', color_continuous_scale='Blues')
fig.update_layout(height=450, yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# Gráfica forecast vs real — tipo de producto con más ventas
df_eval = df_test.copy()
df_eval['pred'] = y_pred
tipo_top = df_eval.groupby('tipo_producto')['unidades'].sum().idxmax()

df_eg = df_eval[df_eval['tipo_producto']==tipo_top]\
    .groupby('semana').agg(real=('unidades','sum'),pred=('pred','sum')).reset_index()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_eg['semana'],y=df_eg['real'],name='Real',line=dict(color='royalblue')))
fig2.add_trace(go.Scatter(x=df_eg['semana'],y=df_eg['pred'],name='Forecast',line=dict(color='orange',dash='dash')))
fig2.update_layout(title=f'Forecast vs Real — {tipo_top} (todas las tiendas)',
                   xaxis_title='Semana',yaxis_title='Unidades')
fig2.show()

## 4 · Generar forecast 8 semanas → Supabase

In [ ]:
fecha_hoy   = df['semana'].max()
fecha_ejec  = date.today()
combinaciones = df[['tienda_id','tipo_producto','familia','ciudad','formato',
                     'segmento_cliente','indice_rotacion','metros_cuadrados']].drop_duplicates()
ultimo = df.sort_values('semana').groupby(['tienda_id','tipo_producto']).last().reset_index()

rows = []
for h in range(1, 9):
    sem_obj = fecha_hoy + pd.Timedelta(weeks=h)
    df_f = combinaciones.copy()
    df_f['semana']       = sem_obj
    df_f['semana_iso']   = sem_obj.isocalendar()[1]
    df_f['mes']          = sem_obj.month
    df_f['anio']         = sem_obj.year
    df_f['trimestre']    = (sem_obj.month-1)//3+1
    df_f['es_quincena']  = int(sem_obj.day in list(range(14,18))+list(range(28,32)))
    df_f['semana_sin']   = np.sin(2*np.pi*df_f['semana_iso']/52)
    df_f['semana_cos']   = np.cos(2*np.pi*df_f['semana_iso']/52)
    df_f['mes_sin']      = np.sin(2*np.pi*df_f['mes']/12)
    df_f['mes_cos']      = np.cos(2*np.pi*df_f['mes']/12)
    df_f = df_f.merge(
        ultimo[['tienda_id','tipo_producto','rolling_4sem','rolling_8sem','rolling_12sem',
                'rolling_std4','tendencia','lag_1sem','lag_2sem','lag_4sem','lag_8sem',
                'lag_52sem','descuento_pct_avg','pct_precio_pleno']],
        on=['tienda_id','tipo_producto'], how='left'
    )
    for col in ['tipo_producto','familia','ciudad','formato','segmento_cliente']:
        df_f[f'{col}_enc'] = encoders[col].transform(df_f[col].astype(str))
    for col in cols_eventos:
        df_f[col] = 0
    df_f = crear_features_eventos(df_f, df_eventos).fillna(0)
    preds = np.maximum(0, modelo.predict(df_f[FEATURES]))
    df_f['forecast_medio']   = preds
    df_f['forecast_bajo']    = preds * 0.80
    df_f['forecast_alto']    = preds * 1.20
    df_f['fecha_ejecucion']  = fecha_ejec
    df_f['semana_objetivo']  = sem_obj.date()
    df_f['error_mape']       = round(mape, 2)
    rows.append(df_f[['fecha_ejecucion','semana_objetivo','tienda_id','tipo_producto',
                       'familia','forecast_bajo','forecast_medio','forecast_alto','error_mape']])

df_forecast = pd.concat(rows, ignore_index=True)
with engine.begin() as conn:
    conn.execute(text("DELETE FROM output_forecast_semanal WHERE fecha_ejecucion = CURRENT_DATE"))
df_forecast.to_sql('output_forecast_semanal',engine,if_exists='append',index=False,method='multi',chunksize=500)
print(f'✅ Forecast guardado: {len(df_forecast):,} filas (8 semanas)')

In [ ]:
os.makedirs('../outputs', exist_ok=True)
with open('../outputs/modelo_lgbm.pkl','wb') as f:
    pickle.dump({'modelo':modelo,'features':FEATURES,'encoders':encoders,'mape':mape,'mae':mae},f)
print('✅ Modelo guardado en outputs/modelo_lgbm.pkl')
print('\n➡️  Continuar con notebook 03_despachos_produccion.ipynb')